# RG3 가스: 이전 샷 상태 전문가

현재 품질 라벨은 유지하고 이전1/3개의 실제 센서 관측 상태6개(Hopper/Barrel2/5/Cushion/PlasticizingTime/PressureSwitchGap)+현재 샷 앞선10개 흔들림4개 총10개를 입력. 센서history_segment 내 shift1/3이며 미래값/앞선 품질 라벨은 미사용. 최소 선행 관측 부족은 학습 정상으로 결측 대체. 라벨 검사 지연/물리 원인을 확인한 것이 아닌 시간 기억 가설.

기존rank75/std4와 lag1/lag3 전문가(각 RF/ET2시드 평균)를 내부F1 기준 단독/쌍평균 선택. 고정 RG3 RH 정상601·가스19/같은7개 관측구간. 새 독립 검증 아닌 개발용 중첩 검증. 과거 센서는 온라인에 관측된다는 가정이며 이전 평가 샷의 센서가 과거 특징에 포함될 수 있으나 그 품질 라벨은 사용하지 않는다. 외부 구간은 결측/분류/앙상블/임계값 학습에서 제외.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
source=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(source.split('F=pd.DataFrame')[0])
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
assert np.array_equal(d.TimeStamp.astype(str),z['timestamps']) and np.array_equal(y,z['labels'])
history_cols=['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']
observed['pressure_switch_gap']=observed.Max_Injection_Pressure-observed.Max_Switch_Over_Pressure
history_cols+=['pressure_switch_gap']
stdcols=[v+'__past10_std' for v in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
keys=['EQUIP_CD','product','TimeStamp'];lag_views={}
for lag in [1,3]:
 names=[v+f'__state_lag{lag}' for v in history_cols]
 observed[names]=observed.groupby('history_segment')[history_cols].shift(lag)
 lagged=d[keys].merge(observed[keys+names],on=keys,validate='one_to_one')[names]
 lag_views[lag]=pd.concat([lagged,d[stdcols]],axis=1).replace([np.inf,-np.inf],np.nan)
 lag_views[lag].assign(TimeStamp=d.TimeStamp).to_csv(t/f'rg3_lag{lag}_state_features.csv',index=False,encoding='utf-8-sig')
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and (y==0).sum()==601
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
memo={};audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];F=z['features_'+n+'_1'];old=z['scores_'+n]
  normal=ref[y[ref]==0]
  values=[old[:,1]]
  for lag in [1,3]:
   im=SimpleImputer(strategy='median',keep_empty_features=True).fit(lag_views[lag].iloc[normal])
   F=im.transform(lag_views[lag]);preds=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
     preds.append(m.fit(F[ref],y[ref]).predict_proba(F)[:,1])
   values.append(np.mean(preds,axis=0))
  values.append(old[:,0])
  memo[key]=np.column_stack(values)
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),normal_imputer_training=len(normal),train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
weights=[np.eye(4)[i] for i in range(4)]+[(np.eye(4)[i]+np.eye(4)[j])/2 for i in range(4) for j in range(i+1,4)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b)
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in weights:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_lag_state',best[5],best[4])]
 for i,name in enumerate(['rank75_reference','lag1_state10','lag3_state10','deep_std4_reference']):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_causal_lag_state_progress.json').write_text(str(int(b)),encoding='utf-8')
 print('Finished lag-state block',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_causal_lag_state_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


Finished lag-state block 1


Finished lag-state block 2


Finished lag-state block 3


Finished lag-state block 4


Finished lag-state block 5


Finished lag-state block 6


Finished lag-state block 7


                           model       AP  TP  FP  FN  TN       F1   recall
deep_std4_reference_inner_F1_cut 0.079209   4  50  15 551 0.109589 0.210526
     inner_F1_selected_lag_state 0.054251   2  46  17 555 0.059701 0.105263
       lag1_state10_inner_F1_cut 0.068280   6 124  13 477 0.080537 0.315789
       lag3_state10_inner_F1_cut 0.043314   2  93  17 508 0.035088 0.105263
   rank75_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
